# HAKE-MER — Step 0 baseline (GoEmotions)

Trains the **flat PLM baseline** (DistilBERT, seeds 42 / 123 / 456): F1-micro, F1-macro, exact match, mAP.

### Open this notebook (repo is private)

The "Open in Colab" badge on GitHub often fails for **private** repos. Use either:

1. **Upload:** [colab.research.google.com](https://colab.research.google.com) → **File → Upload notebook** → pick `baseline_plm_campaign.ipynb` from your local clone.
2. **GitHub in Colab:** **File → Open notebook → GitHub** → sign in → allow Colab → open `khalef-khalil/marii` → `notebooks/baseline_plm_campaign.ipynb`.

### Run

1. **Runtime → Change runtime type → GPU** (T4 is enough).
2. In the clone cell below: if the repo is private, set a Colab secret **`GITHUB_TOKEN`** (key icon in the left sidebar) with a [GitHub token](https://github.com/settings/tokens) that can read this repo, **or** paste the token when prompted.
3. **Runtime → Run all** (about 30–90 minutes on a T4 the first time).

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Use Runtime → Change runtime type → GPU, then run this cell again."
    )
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import subprocess
from getpass import getpass
from pathlib import Path

REPO = "khalef-khalil/marii"
WORKDIR = Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"


def git_token() -> str:
    try:
        from google.colab import userdata

        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return getpass("GitHub token with repo read access (input hidden): ")


def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=True)
        return

    r = subprocess.run(
        ["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)],
        capture_output=True,
    )
    if r.returncode == 0 and (WORKDIR / "run_baseline_campaign.sh").is_file():
        return

    token = git_token()
    if WORKDIR.is_dir():
        subprocess.run(["rm", "-rf", str(WORKDIR)], check=True)
    authed = f"https://{token}@github.com/{REPO}.git"
    subprocess.run(["git", "clone", "--depth", "1", authed, str(WORKDIR)], check=True)


clone_repo()
%cd {WORKDIR}
print("Commit:", end=" ")
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
!./run_baseline_campaign.sh --backbone distilbert-base-uncased

### Optional — RoBERTa-base

Uncomment and run if you need the second backbone from chapter 4.

In [ ]:
# !./run_baseline_campaign.sh --backbone roberta-base

In [ ]:
import json
from pathlib import Path

summary_path = Path(
    "reference/artifacts/baseline_plm_distilbert_base_uncased_campaign.json"
)
if not summary_path.is_file():
    raise FileNotFoundError("Training did not produce the campaign summary. Run the training cell first.")

campaign = json.loads(summary_path.read_text(encoding="utf-8"))
print("Test aggregate (mean ± std over seeds):")
for name, block in campaign["test_aggregate"].items():
    print(f"  {name}: {block['mean']:.4f} ± {block['std']:.4f}")

In [ ]:
import zipfile
from google.colab import files

zip_path = Path("/content/baseline_plm_distilbert_results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(summary_path, summary_path.name)
    for metrics_file in sorted(Path("runs").glob("*_baseline_plm/metrics.json")):
        arcname = f"{metrics_file.parent.name}/{metrics_file.name}"
        zf.write(metrics_file, arcname)

print(f"Zip size: {zip_path.stat().st_size / 1e6:.1f} MB (metrics only, no checkpoints)")
files.download(str(zip_path))